# Продвинутые возможности

## Флаги компиляции

До этого момента мы меняли только шаблон. Но Python позволяет нам менять настройки самого движка регулярных выражений.

Эти настройки назвываются <b>Флагами</b> (Flags). Они работают как глобальные переключатели, которые изменяют поведение поиска для всего выражения сразу.

### Как передавать флаги?

Флаги передаются последним аргументом в функции re.

In [1]:
# re.search(pattern, string, flags=...)
# re.findall(pattern, string, flags=...)

У всех флагов есть два имени:
1. <b>Полное</b>: re.IGNORECASE
2. <b>Короткое</b>: re.I

## Флаг re.IGNORECASE

По умолчанию регулярные выражения в Python регистрозависимы. Чтобы отключить это поведение, используется флаг re.IGNORECASE.

### Пример

Найдем слово "python"

In [2]:
import re

text = "I love PYTHON and PyThOn"
pattern = r"python"

# 1. Без флага (Ничего не найдет)
print(re.findall(pattern, text))
# Вывод: []

# 2. С флагом (Найдет всё)
print(re.findall(pattern, text, flags=re.IGNORECASE))
# Вывод: ['PYTHON', 'PyThOn']

[]
['PYTHON', 'PyThOn']


### Таблетка внутри шаблона: (?i)

Иногда у вас нет доступа к коду Python (например, вы пишете конфиг для валидатора), и вы не можете передавать аргумент flags. Вы можете включить флаг прямо внутри строки регулярного выражения.

Для игнорирования регистра используется конструкция (?i) в начале строки.

In [3]:
# Этот шаблон сам включает режим игнорирования регистра
pattern = r"(?i)python"

match = re.search(pattern, "PYTHON") # Работает!

### Как объединять флаги?

Если вам нужно включить сразу несколько настроек, используйте о<b>ператор битового ИЛИ</b>.

In [4]:
# IGNORECASE + DOTALL
re.search(pattern, text, flags=re.I | re.S)

<re.Match object; span=(7, 13), match='PYTHON'>

## re.DOTALL

Вспомним Модуль 2. Мы говорили, что метасимвол <b>Точка</b> "." - это Джокер, который заменяет любой символ. Но мы сделали важную оговорку:

Точка матчит всё, КРОМЕ переноса строки \n.

### Проблема многострочности

Представьте, что вы парсите комментарий в коде или текст между тегами, который занимает несколько строчек.

In [5]:
text = """
<comment>
   Это важный комментарий.
   Он занимает две строки.
</comment>
"""

Если вы напишете шаллон r'<_comment>(.*)<_/comment>, он ничего не найдет. Почему? Потому что как только движок дойдет до конца первой строки, он встретит невидимый символ \n и прервет поиск.

### Решение: re.DOTALL

Флаг <b>re.DOTALL</b> снимает это ограничение. Он говорит движку:

Теперь точка - это действительно ЛЮБОЙ символ. Даже \n

С этим флагом шаблон .* может проглотить весь документ от начала до конца, сколько бы абзацев в нем ни было.

### Пример в коде

In [6]:
import re

text = """Start
Body
End"""

pattern = r"Start(.*)End"

# 1. По умолчанию (Провал)
# Точка споткнется после слова "Start"
print(re.search(pattern, text))
# Вывод: None

# 2. С флагом DOTALL (Успех)
# Точка съедает \n, Body, \n
match = re.search(pattern, text, flags=re.DOTALL)

if match:
    print(f"Найдено:\n{match.group(1)}")

# Вывод:
# Найдено:
# Body

None
Найдено:

Body



## re.MULTILINE

Мы помним, что якоря "^" и "$" привязываются к границам <b>всего текста</b>.
- ^ - самое начало строки
- $ - самый конец чтроки

Но что, если наш текст - это <b>список</b>, где каждая запись начинается с новой строки?

### Множественные якоря

Флаг <b>re.MULTILINE</b> (или коротко re.M) меняет поведение якорей. С этим флагом:
- <b>^</b> - совпадает с началом текста <b>И</b> сразу после каждого переноса строки \n
- <b>$</b> - совпадает с концом текста <b>И</b> прямо перед каждым переносом строки \n

Теперь якоря срабатывают для каждой строки внутри вашего большого текста.

### Пример

Мы хотим найти всех пользователей в начале строки

In [7]:
import re

text = """user1: admin
user2: guest
error: log
user3: guest"""

# Шаблон: Начало строки, слово user, цифра
pattern = r"^user\d"

# 1. Без флага (Найдем только первого)
print(re.findall(pattern, text))
# Вывод: ['user1']

# 2. С флагом MULTILINE (Найдем всех)
print(re.findall(pattern, text, flags=re.MULTILINE))
# Вывод: ['user1', 'user2', 'user3']

['user1']
['user1', 'user2', 'user3']


## re.VERBOSE

Давайте честно: регулярне выражения выглядят страшно. Когда вы пишете шаблон длиной 50 символов, вы понимаете его только в тот момент, когда пишете.

Если вы вернетесь к этому коду через месяц, вы потратите полчаса, пытаясь расшифровать эту "клинопись". В программировании такой код называют <b>Write-only</b>.

Чтобы сделать регулярки читаемыми, в Python придумали флаг <b>re.VERBOSE</b> (или коротко <b>re.X</b>)

### Что он делает?

Этот флаг кардинально меняет то, как движок читает ваш шаблон:
1. <b>Игнорирует пробелы и переносы строк</b>. Вы можете разбивать регулярку на строчки, делать отступы, отделять логические блоки пробелами. Движок "склеит" всё обратно перед выполнением.
2. <b>Разрешает комментарии</b>. Всё, что идет после знака решетки #, считается комментарием и игнорируется движком.

### До и После

Даавайте перепишем наш страшный парсер даты ISO-8601

<b>Было</b>

In [8]:
pattern = r"^(\d{4})-(\d{2})-(\d{2})T(\d{2}):(\d{2}):(\d{2})Z$"

<b>Стало</b>

Мы используем тройные кавычки r'''...''', чтобы писать в несколько строк:

In [10]:
pattern = r"""
    ^                           # Начало строки
    (\d{4})                     # Год (Группа 1)
    -                           # Разделитель даты
    (\d{2})                     # Месяц (Группа 2)
    -                           # Разделитель даты
    (\d{2})                     # День (Группа 3)
    T                           # Буква T (разделитель времени)
    (\d{2}):(\d{2}):(\d{2})     # Время: ЧЧ:ММ:СС
    Z                           # Часовой пояс UTC
    $                           # Конец строки
"""

# Не забудьте передать флаг!
match = re.search(pattern, "2023-10-05T14:30:00Z", re.VERBOSE)

Согласитесь, второй вариант можно читать как книгу. Такой код не стыдно показать коллегам или оставить потомкам.

## Задачи

### Задача 1. Игнорирование регистра

<b>Условие</b>:

Напишите программу, которая ищет слово debug в тексте, независимо от того, как оно написано: Debug, DEBUG, DeBuG и т.д.
Используйте флаг re.IGNORECASE (или короткий re.I).

<b>Формат ввода</b>:
- Строка.

<b>Формат вывода</b>:
- Found — если слово найдено.
- Not found — если нет.

In [ ]:
import re

pattern = r'\bdebug\b'
text = input()

print(re.search(pattern, text, flags=re.IGNORECASE) and 'Found' or 'Not found')

### Задача 2. Многострочный блок

<b>Условие</b>:

Нам нужно найти текст, заключенный между словами START и END.
Проблема в том, что этот текст может занимать несколько строк.
По умолчанию точка . не видит перенос строки \n.

Используйте флаг re.DOTALL (или re.S), чтобы шаблон START(.*)END сработал.
Выведите содержимое между маркерами.

<b>Формат ввода</b>:
- Многострочный текст.

<b>Формат вывода</b>:
- Найденный текст (или nothing, если не найдено).

In [ ]:
import re
import sys

pattern = r'START(.*)END'
text = sys.stdin.read()

print(re.search(pattern, text, flags=re.DOTALL).group(1).strip())

### Задача 3. Начало каждой строки

<b>Условие</b>:

Вам дан список товаров, каждый с новой строки.
Некоторые строки начинаются с символа - (дефис) — это отсутствующие товары.
Нам нужно найти все такие строки (начинающиеся с дефиса).

Используйте якорь начала строки ^ и флаг re.MULTILINE (или re.M), чтобы ^ срабатывал не только в самом начале текста, но и после каждого переноса строки.

<b>Формат ввода</b>:
- Список товаров.

<b>Формат вывода</b>:
- Список найденных строк (дефис и название).

In [ ]:
import re
import sys

pattern = r'^-\w+'
text = sys.stdin.read()

print(re.findall(pattern, text, flags=re.MULTILINE))

### Задача 4. Комбо флагов

<b>Условие</b>:

Иногда нужно всё и сразу.
Найдите содержимое тега <_div ...> ... <_/div>.
Условия:
1. Тег может быть написан в любом регистре: <_DIV>, <_div>, <_DiV>. (Нужен re.I)
2. Содержимое может быть многострочным. (Нужен re.S)

Используйте оператор | для объединения флагов.
Шаблон: <_div.*?>(.*?)<_/div> (используйте ленивые квантификаторы).

<b>Формат ввода</b>:
- HTML код.

<b>Формат вывода</b>:
- Содержимое тега.

In [ ]:
import re
import sys

pattern = r'''
    ^
    <div.*>
    (.*)
    </div>
    $
'''
text = sys.stdin.read()
match = re.search(pattern, text, flags=re.IGNORECASE|re.DOTALL|re.VERBOSE)

print(match and match.group(1).strip() or '')

### Задача 5. Читаемый шаблон

<b>Условие</b>:

Нам нужно найти цвет в формате HEX (например, #FFFFFF).
Чтобы шаблон был читаемым, мы разбили его на части и добавили пробелы.
Но теперь он не работает, потому что пробелы считаются частью поиска!

Добавьте флаг <b>re.VERBOSE</b> (или re.X), чтобы Python игнорировал пробелы и комментарии в шаблоне.

<b>Дан шаблон</b>:

In [11]:
pattern = r"""
    \#          # Символ решетки
    [0-9a-fA-F] # Цифра или буква от A до F
    {6}         # Ровно 6 раз
"""

Вам нужно использовать его в re.search.

<b>Формат ввода</b>:
- Строка с CSS кодом.

<b>Формат вывода</b>:
- Найденный цвет или No color.

In [ ]:
import re 

text = input()
match = re.search(pattern, text, flags=re.VERBOSE)

print(match and match.group() or 'No color')

## Positive Lookahead

До этого момента движок регулярных выражений работал как Пакман: он шел по строке и "съедал" символы, добавляя их в результат.

Но иногда нам нужно <b>проверить условие</b>, не "съедая" текст. Например: "Найди имя Isaac, но только если после него идет фамилия Newton. Но саму фамилию в результат не включай, мне нужно только имя".

Для этого используется <b>Опередающие проверки (Lookahead)</b>.

### Синтаксис

Позитиваня опережающая проверка записывается как группа, начинающаяся с ?=:

<b>A(?=B) - найди А, при условии, что сразу за ним следует В</b>.

Главная фишка: В проверятся, но не добавляется в результат (и не потребляется движком). Курсор остается стоять сразу после А.

### Пример 1. Имена файлов

Представьте, что у нас есть список файлов: script.py, text.txt, main.py, image.jpg.

<b>Задача</b>: получить имена только тех файлов, которые имеют расширение .py. При этом само расширение .py нам не нужно.

In [14]:
import re

files = "script.py text.txt main.py image.jpg"

# Шаблон:
# 1. \w+       - Найти слово (Имя файла)
# 2. (?=\.py)  - Проверить, что ВПЕРЕДИ стоит точка и py
pattern = r"\w+(?=\.py)"

print(re.findall(pattern, files))
# Вывод: ['script', 'main']

['script', 'main']


<b>Разбор полетов</b>
1. Движок находит script
2. Смотрит вперед (Lookahead)
3. Условие выполнено
4. Движок возвращает script
5. <b>Важно</b>: Курсор останавливается перед точкой. Расширение .py остается в тексте нетронутым (хотя мы его проверили)

## Negative Lookahead

В прошлом уроке мы научились заглядывать вперед и говорить: "Я возьму это, только если дальше стоит БОНУС". Теперь мы научимся быть подозрительными: "Я возьму это, только если дальше НЕТ подвоха".

Это называется <b>Негативная опережающая проверка (Negative Lookahead)</b>.

### Синтаксис (?!...)

Синтасис почти такой же, но вместо знака равенства ставится восклицательный знак (который в программировании традиционно означает "НЕ").

<b>A(?!B) - найди А, при условии, что сразу за ним НЕ следует В.</b>

Если движок видит В, он сбрасывает совпадение для А.

### Пример 1. Нежелательные персоны

Представьте, что мы ищем в тексте всех Джонов, но нам не нужен Джон Доу, так как это неизвестная личность. Все остальные Джоны нам подходят.

In [15]:
import re

names = "John Smith, John Doe, John Wayne"

# Шаблон:
# 1. John  - Находим имя
# 2. (?! Doe) - ПРОВЕРЯЕМ: А не " Doe" ли там дальше?
pattern = r"John(?! Doe)"

matches = re.findall(pattern, names)
print(matches)
# Вывод: ['John', 'John']

['John', 'John']


### Пример 2. Корпоративная почта

Задача: найти все email-адреса в тексте, кроме адресов техподдержки

In [16]:
import re

emails = "admin@site.com, support@site.com, user@site.com"

# \b        - Граница слова (начало)
# (?!support) - Сразу проверяем: это НЕ support?
# \w+       - Если проверка прошла, матчим имя
# @site.com - Остальная часть
pattern = r"\b(?!support)\w+@site\.com"

print(re.findall(pattern, emails))
# Вывод: ['admin@site.com', 'user@site.com']

['admin@site.com', 'user@site.com']


## Lookbehind

Мы научились смотреть вперед (Lookahead). Теперь давайте обернемся назад. <b>Ретроспективная проверка (Lookbehind)</b> позволяет найти текст, основываясь на том, что стоит <b>перед</b> ним (слева).

### Positive Lookbehind (?<=...)

Синтаксис похож на Lookahead, но добавляется знак меньше "<" (стрелочка влево).

<b>(?<=B)A - найди А, если сразу перед ним (слева) стоит В.</b>

Как и раньше, В проверяется, но не попадает в результат.

### Пример. Ценники без валюты

Нам нужны только цифры, но мы должны знать, какая валюта стоит перед ними.

In [17]:
import re

text = "Price: $100, Cost: €20"

# 1. Найти цифры, перед которыми стоит знак доллара
# (?<=\$) - Оглянись назад. Там доллар?
# \d+     - Если да, бери цифры.
print(re.findall(r"(?<=\$)\d+", text))
# Вывод: ['100']

# 2. Найти цифры, перед которыми стоит евро
print(re.findall(r"(?<=€)\d+", text))
# Вывод: ['20']

['100']
['20']


### Negative Lookbehind (?<!...)

Синтасис: стрелочка влево < и восклицательный знак !

<b>(?<!B)A - найди А, если сразу перед ним НЕТ В.</b>

### Боль и слезы Python: Fixed-width limitation

Здесь кроется главное разочарование модуля re в Python. Движок Python требует, чтобы <b>шаблон внутри Lookbehind имел ФИКСИРОВАННУЮ длину</b>.

Движок должен точно знать, на сколько шагов назад ему нужно отпрыгнуть, чтобы начать проверку. Он не умеет отпрыгивать на "неизвестное количество" шагов.

## Задачи

### Задача 1. Только евро

<b>Условие</b>:

Вам дан текст с ценами в разных валютах.
Нужно найти только числа, которые относятся к евро (€).
Если число стоит перед знаком доллара ($) или без знака, оно нам не нужно.

Напишите шаблон, который ищет цифры, за которыми сразу следует символ €. Сам символ € в результат попасть не должен.
Используйте re.findall.

<b>Формат ввода</b>:
- Строка с ценами.

<b>Формат вывода</b>:
- Список чисел (строк).

In [ ]:
import re

pattern = r'\d+(?=€)'
text = input()

print(re.findall(pattern, text))

### Задача 2. Android не 4 версии

<b>Условие</b>:

Мы ищем упоминания операционной системы Android.
Но нас не интересует старая версия Android 4. Все остальные (Android 9, Android 13, просто Android) нам подходят.

Напишите регулярное выражение, которое находит слово Android, но только если сразу после него НЕТ пробела и цифры 4.

<b>Формат ввода</b>:
- Строка.

<b>Формат вывода</b>:
- Список найденных совпадений.

In [ ]:
import re

pattern = r'Android(?!\s4)'
text = input()

print(re.findall(pattern, text))

### Задача 3. Хэштег без решетки

<b>Условие</b>:

Обычно мы ищем хэштеги так: #\w+. Но тогда в результат попадает и сама решетка #.
Используя Positive Lookbehind, найдите слово, перед которым стоит решетка, но саму решетку в результат не включайте.

Шаблон: "Оглянись назад, там решетка? Если да, бери слово".
Напоминание: в Python lookbehind должен быть фиксированной длины.

<b>Формат ввода</b>:
- Текст с хэштегами.

<b>Формат вывода</b>:
- Список слов (без #).

In [ ]:
import re

pattern = r'(?<=#)\w+'
text = input()

print(re.findall(pattern, text))

### Задача 4. Не цена

<b>Условие</b>:

У нас есть строка с данными: Qty: 5, Price: $100.
Нам нужно найти число, которое НЕ является ценой (то есть перед ним НЕТ знака доллара $).

Используйте Negative Lookbehind.
Также используйте границу слова \b перед числом, чтобы случайно не найти 00 внутри 100.

<b>Формат ввода</b>:
- Строка.

<b>Формат вывода</b>:
- Список найденных чисел.

In [ ]:
import re

pattern = r'\b(?<!\$)\d+'
text = input()

print(re.findall(pattern, text))

### Задача 5. Сложность пароля

<b>Условие</b>:

Это классическая задача. Проверьте, является ли пароль надежным.
Критерии надежности (упрощенные):
1. Содержит хотя бы одну цифру.
2. Содержит хотя бы одну строчную букву.
3. Длина от 6 символов.

Используйте re.search и конструкцию из двух Lookahead в начале строки:
(?=.*\d) — проверка на цифру где-то в строке.
(?=.*[a-z]) — проверка на букву где-то в строке.
И затем проверка длины .{6,}.

<b>Формат ввода</b>:
- Пароль.

<b>Формат вывода</b>:
- Strong или Weak.

In [ ]:
import re

pattern = r'(?=.*\d)(?=.*[a-z]).{6,}'
text = input()

print(re.search(pattern, text) and 'Strong' or 'Weak')